In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm 
import joblib

In [2]:
# Se cargan los datos. 

data=pd.read_csv('car_details_v3.csv',sep=',')

In [3]:
# Cantidad de datos y número de variables

data.shape

(8128, 13)

In [4]:
# Mostrar los datos

data.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [5]:
# Es recomendable que todos los pasos preparación se realicen sobre otro archivo.

data_t = data

In [6]:
# Podemos comprobar las ausencias con esta línea de código. 

data_t.isna().sum()/len(data_t)

name             0.000000
year             0.000000
selling_price    0.000000
km_driven        0.000000
fuel             0.000000
seller_type      0.000000
transmission     0.000000
owner            0.000000
mileage          0.027190
engine           0.027190
max_power        0.026452
torque           0.027313
seats            0.027190
dtype: float64

In [7]:
# Eliminación de registros con ausencias.

data_t=data_t.dropna()

In [25]:
#se elimina texto de las variables numericas

data_t['mileage'] = data_t['mileage'].astype(str).str.extract(r'([\d.]+)').astype(float)
data_t['engine'] = data_t['engine'].astype(str).str.extract(r'([\d.]+)').astype(float)
data_t['max_power'] = data_t['max_power'].astype(str).str.extract(r'([\d.]+)').astype(float)

In [17]:
# Podemos ver el tamaño del conjunto de datos después de este paso de limpieza de datos.

data_t.shape

(6717, 20)

In [18]:
# Eliminación de registros duplicados.

data_t=data_t.drop_duplicates()

In [19]:
# Podemos ver el tamaño del conjunto de datos después de este paso de limpieza de datos.

data_t.shape

(6717, 20)

In [20]:
# Se muestran las categorías de la variable "propietario" con sus frecuencias

pd.Series.value_counts(data['owner'])

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

In [21]:
# Se muestran las categorías de la variable "combustible" con sus frecuencias

pd.Series.value_counts(data['fuel'])

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

In [22]:
# Se realiza la transformación de estas variables a dummies.

data_t = pd.get_dummies(data_t, columns=['fuel','owner'],dtype=int)

KeyError: "None of [Index(['fuel', 'owner'], dtype='object')] are in the [columns]"

In [23]:
# Cantidad de datos y número de variables después de esta transformación.

data_t.shape

(6717, 20)

In [26]:
# Ahora el conjunto de datos tiene 20 variables.Veamos las primeras filas de este nuevo conjunto de datos.

data_t.head()

,name,year,selling_price,km_driven,seller_type,transmission,mileage,engine,max_power,torque,seats,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti Swift Dzire VDI,2014,450000,145500,Individual,Manual,23.40,1248.0,74.00,190Nm@ 2000rpm,5.0,0,1,0,0,1,0,0,0,0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Individual,Manual,21.14,1498.0,103.52,250Nm@ 1500-2500rpm,5.0,0,1,0,0,0,0,1,0,0
2,Honda City 2017-2020 EXi,2006,158000,140000,Individual,Manual,17.70,1497.0,78.00,"12.7@ 2,700(kgm@ rpm)",5.0,0,0,0,1,0,0,0,0,1
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Individual,Manual,23.00,1396.0,90.00,22.4 kgm at 1750-2750rpm,5.0,0,1,0,0,1,0,0,0,0
4,Maruti Swift VXI BSIII,2007,130000,120000,Individual,Manual,16.10,1298.0,88.20,"11.5@ 4,500(kgm@ rpm)",5.0,0,0,0,1,1,0,0,0,0


In [34]:
# Eliminación de filas no numericas.

data_t=data_t.drop(['name'], axis=1)
data_t=data_t.drop(['seller_type'], axis=1)
data_t=data_t.drop(['transmission'], axis=1)
data_t=data_t.drop(['torque'], axis=1)

In [35]:
# Se selecciona la variable objetivo, en este caso "selling_price".

Y=data_t['selling_price']
X=data_t.drop(['selling_price'], axis=1)

In [36]:
# Mostramos nuestros datos

X.head()

,year,km_driven,mileage,engine,max_power,seats,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,2014,145500,23.40,1248.0,74.00,5.0,0,1,0,0,1,0,0,0,0
1,2014,120000,21.14,1498.0,103.52,5.0,0,1,0,0,0,0,1,0,0
2,2006,140000,17.70,1497.0,78.00,5.0,0,0,0,1,0,0,0,0,1
3,2010,127000,23.00,1396.0,90.00,5.0,0,1,0,0,1,0,0,0,0
4,2007,120000,16.10,1298.0,88.20,5.0,0,0,0,1,1,0,0,0,0


In [37]:
Y.head()

0    450000
1    370000
2    158000
3    225000
4    130000
Name: selling_price, dtype: int64

In [38]:
# Se realiza la división entrenamiento - test. Se deja 20% de los datos para el test.

X_train, X_test, Y_train, Y_test = train_test_split( X, Y, test_size=0.2, random_state=0)

In [39]:
# Primero se crea el objeto para construir el modelo

modelo_regresion = LinearRegression()
modelo_regresion

LinearRegression()

In [40]:
# Ajustar el modelo con los datos de entrenamiento

modelo_regresion.fit(X_train,Y_train)

LinearRegression()

In [42]:
# Se obtienen las predicciones del modelo sobre el conjunto de entrenamiento.

y_pred = modelo_regresion.predict(X_train)

print("MSE: %.2f" % mean_squared_error(Y_train, y_pred))
print("MAE: %.2f" % mean_absolute_error(Y_train, y_pred))
print('R²: %.2f' % r2_score(Y_train, y_pred))

MSE: 99628182480.44
MAE: 174590.80
R²: 0.62


In [44]:
# Se obtienen las predicciones del modelo sobre el conjunto test.

y_pred = modelo_regresion.predict(X_test)

print("MSE: %.2f" % mean_squared_error(Y_test, y_pred))
print("MAE: %.2f" % mean_absolute_error(Y_test, y_pred))
print('R²: %.2f' % r2_score(Y_test, y_pred))

MSE: 129372479578.93
MAE: 181378.90
R²: 0.60


In [45]:
# Ajustar el modelo con los datos de entrenamiento

modelo_regresion.fit(X,Y)

LinearRegression()

In [46]:
# Podemos visualizar los parámetros del modelos (coeficientes de regresión)

modelo_regresion.coef_

array([ 3.48725938e+04, -7.72476031e-01,  2.45410959e+03,  8.54383576e+01,
        9.73803389e+03, -2.18364396e+04, -3.46329107e+04, -4.54700661e+03,
        9.34160710e+04, -5.42361537e+04, -5.39957871e+05, -5.87592439e+05,
       -6.06476491e+05,  2.34450985e+06, -6.10483048e+05])

In [47]:
# Ajustar el modelo para ver el reporte

model = sm.OLS(Y, X).fit() 
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          selling_price   R-squared:                       0.617
Model:                            OLS   Adj. R-squared:                  0.616
Method:                 Least Squares   F-statistic:                     829.2
Date:                Thu, 26 Mar 2026   Prob (F-statistic):               0.00
Time:                        17:22:08   Log-Likelihood:                -94763.
No. Observations:                6717   AIC:                         1.896e+05
Df Residuals:                    6703   BIC:                         1.896e+05
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
year                        3.487e+04   1422.206     24.520      0.000    3.21e+04    3.77e+04
km_driven                     -0.7725      0.079     -9.814      0.000      -0.927      -0.618
mileage                     2454.1096   1718.770      1.428      0.153    -915.226    5823.445
engine                        85.4384     18.896      4.522      0.000      48.397     122.480
max_power                   9738.0339    192.938     50.472      0.000    9359.814    1.01e+04
seats                      -2.184e+04   5964.627     -3.661      0.000   -3.35e+04   -1.01e+04
fuel_CNG                    -3.89e+07   1.58e+06    -24.651      0.000    -4.2e+07   -3.58e+07
fuel_Diesel                -3.887e+07   1.58e+06    -24.663      0.000    -4.2e+07   -3.58e+07
fuel_LPG                   -3.877e+07   1.58e+06    -24.589      0.000   -4.19e+07   -3.57e+07
fuel_Petrol                -3.892e+07   1.58e+06    -24.672      0.000    -4.2e+07   -3.58e+07
owner_First Owner          -3.163e+07   1.26e+06    -25.034      0.000   -3.41e+07   -2.92e+07
owner_Fourth & Above Owner -3.168e+07   1.26e+06    -25.193      0.000   -3.41e+07   -2.92e+07
owner_Second Owner          -3.17e+07   1.26e+06    -25.143      0.000   -3.42e+07   -2.92e+07
owner_Test Drive Car       -2.875e+07   1.27e+06    -22.579      0.000   -3.12e+07   -2.63e+07
owner_Third Owner           -3.17e+07   1.26e+06    -25.179      0.000   -3.42e+07   -2.92e+07
==============================================================================
Omnibus:                     7155.866   Durbin-Watson:                   1.698
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          1064155.410
Skew:                           5.087   Prob(JB):                         0.00
Kurtosis:                      63.817   Cond. No.                     6.67e+20
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 1.34e-28. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [49]:
# Ahora guardamos el modelo con el nombre que selecciones.

joblib.dump(modelo_regresion, 'ModeloRegresion.joblib')
# modelo = load('ModeloRegresion.joblib') 

['ModeloRegresion.joblib']